# Enterprise RAG — Hands-On, Part 3 of 11: Compiling the policy into a database filter

*Split from `02-hands-on.ipynb` for focused reading — same content, one phase at a time. The
"Setup" cell below re-derives whatever state earlier parts would have produced, so this notebook
runs standalone; you do not need to run the other parts first.*

**Prerequisites:** `OPENAI_API_KEY` in the repo-root `.env`, and `python scripts/ingest.py` already
run (the setup cell below will build the index for you if it is missing).

**Series:** [1. The corpus and its permissions](part01-corpus-and-permissions.ipynb) · [2. The policy engine](part02-policy-engine.ipynb) · [3. Compiling the policy into a database filter](part03-compiling-policy-to-filter.ipynb) · [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb) · [5. Why hybrid search, demonstrated](part05-hybrid-search.ipynb) · [6. Query transformation](part06-query-transformation.ipynb) · [7. Reranking](part07-reranking.ipynb) · [8. The full graph](part08-full-graph.ipynb) · [9. Attacking it](part09-attacking-it.ipynb) · [10. Evaluation](part10-evaluation.ipynb) · [11. Observability, and what to take away](part11-observability-and-takeaways.ipynb)

---


In [ ]:
import sys, json, textwrap
from pathlib import Path

# The package lives in src/ - add it to the path so this notebook runs from anywhere.
ROOT = Path.cwd()
while not (ROOT / "src" / "enterprise_rag").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from enterprise_rag.config import SETTINGS

print("project root :", ROOT)
print("corpus       :", SETTINGS.corpus_dir.relative_to(ROOT))
print("api key      :", "found" if SETTINGS.has_api_key else "MISSING - check .env")
print("embed model  :", SETTINGS.embedding_model)
print("chat model   :", SETTINGS.fast_model)

### Setup — recap of state from earlier parts


In [ ]:
from enterprise_rag.identity import get_principal

---
# Part 3 - Compiling the policy into a database filter

The policy engine is expressive. A vector database's filter language is not - Chroma cannot even
store a list value. So we **compile** the statically-decidable part of the policy into a `where`
clause and push it down into the search.

**The list problem and its fix:** group membership is encoded as *one boolean column per group*
(`grp__engineering: True`), so an `$or` of `$eq True` reproduces list-overlap semantics.

In [ ]:
from enterprise_rag.authz.policy import compile_prefilter, explain_prefilter

p = get_principal("u_marco_t3")
print(explain_prefilter(p), "\n")
print(json.dumps(compile_prefilter(p), indent=2))

Now the critical part - **what deliberately does *not* go into the filter:**

| Pushed down (layer 1) | Kept for the post-check (layer 2) | Why |
|---|---|---|
| tenant | embargo / expiry | needs "now"; a stale index would leak an unpublished doc |
| clearance level | need-to-know compartments | list semantics the DB can't express |
| region | live revocation | group membership may have changed since indexing |
| group overlap | PII redaction | it's a *transformation*, not a filter |

> **The filter makes retrieval cheap. The post-check makes it correct.**

We can see the gap directly - Erin's *retrievable pool* contains advisory chunks, but the policy
denies her both advisory documents. Layer 2 is what closes that gap.

In [ ]:
from enterprise_rag.ingest.store import fetch_all_allowed
from collections import defaultdict

for uid in ["u_lena_t1", "u_marco_t3", "u_sofia_am", "u_erin_secmgr", "u_attacker_other_tenant"]:
    pp = get_principal(uid)
    pool = fetch_all_allowed(pp.tenant_id, compile_prefilter(pp))
    by_src = defaultdict(int)
    for c in pool:
        by_src[c.attrs.source] += 1
    print(f"{uid:<26}{len(pool):>3} chunks   {dict(sorted(by_src.items())) or 'nothing'}")

print("\nNote Erin: 9 advisory chunks survive the PRE-FILTER,")
print("but the policy denies her both advisories (embargo + need-to-know).")
print("That gap is exactly what the post-retrieval enforcement layer exists to close.")

---

**◀ Previous:** [2. The policy engine](part02-policy-engine.ipynb)

**Next ▶:** [4. Chunking and ingestion](part04-chunking-and-ingestion.ipynb)
